# DART 사업의 내용 추출 결과 정리: 최종본 만들기

`dart_business_extractor.ipynb`가 회사마다 따로 받아 둔 LLM 결과(Codex·배치)를 **하나의 파일로 합치고, 규칙으로 고쳐 최종본을 만든다.** 원칙은 `graphDB.md` 5장(5.4)을 따른다.

- 원본 결과(`extract/raw/v5/`)는 건드리지 않는다. 최종본에서 고친 것은 모두 변경 기록(`changes.csv`)에 남기므로, 원본을 다시 찾을 수 있다.
- 서버나 LLM에 요청하지 않는다. 몇 번을 다시 실행해도 된다.

**고치는 것**

| 번호 | 내용 | 처리 |
|---|---|---|
| ① | 근거가 원문과 다름(표 숫자를 잘못 옮김 등) | 원문에서 가장 비슷한 부분으로 바꾼다. 비슷한 부분이 없으면 그 근거와 그 근거를 쓰는 항목을 뺀다 |
| ② | 자기 자신을 상대로 적은 관계 | 뺀다 |
| ③ | 기타 관계 중 지분 관계(자회사·관계회사·출자·주주)만 적은 것 | 뺀다. 지분 관계는 `dart_parser` 결과를 쓴다. 사업 내용(공급·개발 등)이 적혀 있으면 남긴다 |
| ④ | 국내 사업장 단위 상대(정비사업조합, PFV, 부동산투자회사 등) | 유형이 분명한 것만 뺀다 |
| ⑤ | 이름 띄어쓰기 깨짐("전 장"), 같은 항목 중복 | 붙여 쓰고, 같은 것은 하나만 남긴다 |

**입력**

| 파일 | 내용 |
|---|---|
| `source/dart/business/extract/runs.csv` | 실행 기록. 회사마다 성공한 마지막 결과를 쓴다 |
| `source/dart/business/extract/raw/v5/` | 원본 결과(Codex·배치) |
| `source/dart/business/inputs/종목코드.txt` | LLM 입력 글(근거 확인용) |
| `source/dart/parsed/*.csv` | 기업 이름, 지분 관계(`dart_parser`) |

**출력** (`source/dart/business/final/`)

| 파일 | 내용 |
|---|---|
| `dart_business_v5.json` | 최종본. 회사 1,063개, 구조는 지시문 스키마와 같고 회사 이름·실행 수단을 덧붙인다 |
| `changes.csv` | 고친 것 한 줄: 회사, 번호(①~⑤), 항목, 바꾸기 전, 바꾼 뒤, 이유 |

**걸리는 시간**: 약 1분.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `dart_business_extractor` → **`dart_business_cleaner`** → `name_unifier` |
| 외부 요청·비용 | 없음 |
| 다시 실행 | 언제든 된다. 원본 결과는 고치지 않고 매번 처음부터 최종본을 다시 만든다. 내용이 바뀔 때만 파일을 쓴다 |
| 멈추는 경우 | 결과가 없거나 입력 글이 바뀐 회사가 있을 때(2절), 최종본에 원문에 없는 근거가 남을 때(11절) |

## 1. 설정

In [1]:
# ── 설정 ──
import difflib
import hashlib
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'

PROMPT_VERSION = 'v5'
MODEL = 'gpt-6.1-terra'
BUSINESS = ROOT / 'source' / 'dart' / 'business'
EXTRACT = BUSINESS / 'extract'
INPUTS = BUSINESS / 'inputs'
PARSED = ROOT / 'source' / 'dart' / 'parsed'
FINAL = BUSINESS / 'final'
FINAL.mkdir(parents=True, exist_ok=True)

LISTS = ['products', 'brands', 'materials', 'customers', 'competitors', 'market_shares', 'contracts',
         'price_items', 'other_relations']
squash = lambda s: re.sub(r'\s+', '', s or '')

## 2. 결과 모으기

- `runs.csv`에서 지시문 v5·`gpt-6.1-terra`로 **성공한** 실행만 고르고, 회사마다 마지막 것을 쓴다. Codex로 돌렸든 배치로 돌렸든 같은 지시문·모델이면 같은 결과로 본다.
- 결과가 만들어질 때의 입력 파일과 지금 입력 파일이 같은지(`input_hash`) 확인한다. 다르면 입력이 바뀐 뒤 다시 돌리지 않은 것이라 멈춘다.

In [2]:
# ── 결과 모으기: 회사마다 v5로 성공한 마지막 결과 ──
def parse_result(resp):
    """원본 응답에서 결과 JSON을 꺼낸다(배치 한 줄, API, Codex)."""
    if 'custom_id' in resp:
        resp = (resp.get('response') or {}).get('body') or {}
    if isinstance(resp.get('output'), list):
        text = next((c.get('text') or '' for item in resp['output'] if item.get('type') == 'message'
                     for c in item.get('content') or [] if c.get('type') == 'output_text'), '')
    else:
        text = resp.get('last_message') or resp.get('result') or ''
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    return json.loads(text[text.find('{'):])


companies = pd.read_csv(PARSED / 'companies.csv', dtype=str).fillna('').set_index('stock_code')
runs = pd.read_csv(EXTRACT / 'runs.csv', dtype=str)
ok = runs[(runs['status'] == 'ok') & (runs['prompt_version'] == PROMPT_VERSION) & (runs['model'] == MODEL)]
ok = ok.sort_values('run_utc').drop_duplicates('stock_code', keep='last').set_index('stock_code')

codes = sorted(p.stem for p in INPUTS.glob('*.txt') if re.fullmatch(r'[0-9A-Z]{6}', p.stem))
texts = {c: (INPUTS / f'{c}.txt').read_text(encoding='utf-8') for c in codes}
missing = [c for c in codes if c not in ok.index]
stale = [c for c in codes if c in ok.index and ok.at[c, 'input_hash'] != hashlib.sha256(texts[c].encode('utf-8')).hexdigest()[:16]]
assert not missing and not stale, f'결과 없음 {missing[:5]}, 입력 바뀜 {stale[:5]}'

results, raw_counts = {}, {}
for c in codes:
    r = ok.loc[c]
    folder = 'codex-gpt-6.1-terra' if r['runner'] == 'codex' else 'api-gpt-6.1-terra'
    out = parse_result(json.loads((EXTRACT / 'raw' / PROMPT_VERSION / folder / r['raw_file']).read_text(encoding='utf-8')))
    results[c] = out
    raw_counts[c] = {**{k: len(out[k]) for k in LISTS}, 'materials.suppliers': sum(len(m['suppliers']) for m in out['materials'])}

print('회사', len(results), '/ 실행 수단', ok.loc[codes, 'runner'].value_counts().to_dict())

회사 1063 / 실행 수단 {'batch': 963, 'codex': 98, 'api': 2}


## 3. 변경 기록 준비

고칠 때마다 `log()`로 한 줄씩 남긴다. 항목을 뺄 때도 무엇을 뺐는지 적는다.

In [3]:
# ── 변경 기록: 고치거나 뺄 때마다 한 줄 ──
changes = []


def log(code, rule, section, before, after, why):
    """변경 기록에 한 줄을 더한다(회사, 규칙 번호, 목록, 바꾸기 전, 바꾼 뒤, 이유)."""
    changes.append(dict(stock_code=code, stock_name=companies.at[code, 'stock_name'], rule=rule, section=section,
                        before=str(before)[:300], after=str(after)[:300], why=why))


def item_label(section, x):
    """변경 기록에 남길 항목 이름."""
    for k in ('name', 'item', 'counterparty', 'product'):
        if x.get(k):
            return x[k]
    return ''

## 4. ① 근거 바로잡기

근거(`evidence`)는 입력 글을 그대로 옮긴 것이어야 한다(공백 차이는 무시). 다르면:

1. 입력 글의 줄마다, 근거와 길이가 비슷한 부분을 잘라 **글자 일치율**을 잰다.
2. 가장 높은 일치율이 0.85 이상이면 근거를 그 원문 부분으로 바꾼다. 대부분 표의 숫자 한두 자리를 잘못 옮긴 경우다.
3. 0.85 미만이면 원문에서 찾을 수 없는 근거로 보고, 그 근거와 그 근거를 쓰는 항목을 뺀다.
4. 근거를 바꾼 항목의 숫자 표기(`share_text` 등)가 원문에 없으면 잘못 옮긴 숫자일 수 있어 빈 글자로 둔다.

In [4]:
# ── ① 근거 바로잡기: 원문과 다른 근거는 비슷한 원문으로 바꾸고, 못 찾으면 그 근거를 쓰는 항목째 뺀다 ──
# 이 일치율 이상이면 같은 문장을 옮기다 틀린 것으로 보고 원문으로 바꾼다
MATCH_MIN = 0.85
TEXT_FIELDS = ['share_text', 'amount_text', 'period_text', 'change_text']


def best_match(evidence, text):
    """입력 글에서 근거와 가장 비슷한 부분과 일치율."""
    e = squash(evidence)
    # 근거의 6글자 조각이 하나도 없는 줄은 건너뛴다(속도)
    grams = {e[i:i + 6] for i in range(0, max(1, len(e) - 5), 3)}
    best = (0.0, '')
    for line in text.splitlines():
        flat = squash(line)
        if not flat or not any(g in flat for g in grams):
            continue
        words = line.split()
        if len(flat) <= len(e) * 1.15:
            spans = [line.strip()]
        else:   # 줄이 길면 근거 길이만큼 단어 단위로 잘라 본다
            spans = []
            for i in range(len(words)):
                acc = ''
                for j in range(i, len(words)):
                    acc += words[j]
                    if len(acc) > len(e) * 1.15:
                        break
                    if len(acc) >= len(e) * 0.9:
                        spans.append(' '.join(words[i:j + 1]))
        for span in spans:
            ratio = difflib.SequenceMatcher(None, e, squash(span), autojunk=False).ratio()
            if ratio > best[0]:
                best = (ratio, span)
    return best


def items_using(out, eid):
    """근거 번호 eid를 쓰는 항목들을 (목록 이름, 항목)으로 돌려준다."""
    return [(k, x) for k in LISTS for x in out[k] if x.get('evidence_id') == eid]


fixed = dropped = 0
for c, out in results.items():
    src = squash(texts[c])
    for ev in list(out['evidence']):
        if squash(ev['text']) in src:
            continue
        ratio, span = best_match(ev['text'], texts[c])
        if ratio >= MATCH_MIN:
            log(c, '①', 'evidence', ev['text'], span, f'원문과 다른 근거를 원문으로 바꿈(일치율 {ratio:.2f})')
            ev['text'] = span
            fixed += 1
            for k, x in items_using(out, ev['id']):
                for f in TEXT_FIELDS:
                    if x.get(f) and squash(x[f]) not in src:
                        log(c, '①', k, f'{item_label(k, x)}: {x[f]}', '', f'{f}가 원문에 없어 비움')
                        x[f] = ''
        else:
            for k, x in items_using(out, ev['id']):
                out[k].remove(x)
                log(c, '①', k, item_label(k, x), '', f'근거를 원문에서 찾을 수 없어 항목을 뺌(일치율 {ratio:.2f})')
            out['evidence'].remove(ev)
            log(c, '①', 'evidence', ev['text'], '', '원문에서 찾을 수 없는 근거')
            dropped += 1

print(f'근거 바꿈 {fixed}개, 뺌 {dropped}개')

근거 바꿈 29개, 뺌 1개


## 5. 이름 비교 규칙

같은 회사인지 볼 때는 법인 표기(`(주)`, `㈜`, `주식회사`, `Co., Ltd.`, `Inc.` 등), 띄어쓰기, 문장부호를 지우고 소문자로 비교한다. 예: `고려아연㈜` = `고려아연`.

In [5]:
# ── 이름 비교 키: 법인 표기·띄어쓰기·문장부호를 지운다 ──
def norm_name(s):
    """같은 회사인지 비교하는 키."""
    s = (s or '').lower()
    s = re.sub(r'\(주\)|㈜|주식회사|\(유\)|유한회사|\(株\)|株式會社', '', s)
    s = re.sub(r'\b(co|ltd|inc|corp|corporation|limited|company|llc|plc)\b\.?', '', s)
    return re.sub(r'[\s.,·\-()&]', '', s)


print(norm_name('고려아연㈜'), norm_name('SAMSUNG ELECTRONICS CO,.LTD'), norm_name('Tesla, Inc.'))

고려아연 samsungelectronics tesla


## 6. ② 자기 자신 빼기

매입처·고객·경쟁사·계약 상대·기타 관계의 상대가 **그 회사 자신**(종목명, 법인명, 영문 법인명)이면 뺀다. 예: 고려아연의 고객 "고려아연㈜". 사업보고서가 회사 안의 부문 간 거래를 적은 경우다.

In [6]:
# ── ② 자기 자신 빼기: 상대가 그 회사 자신인 항목 ──
def counterparties(out):
    """(목록 이름, 항목, 상대 이름 키, 원재료 항목) — 상대가 있는 모든 자리."""
    for m in out['materials']:
        for s in m['suppliers']:
            yield 'materials.suppliers', s, 'name', m
    for k, key in [('customers', 'name'), ('competitors', 'name'), ('contracts', 'counterparty'),
                   ('other_relations', 'counterparty')]:
        for x in out[k]:
            yield k, x, key, None


def remove(out, section, x, parent):
    """항목을 그 목록에서 뺀다(매입처는 원재료 항목 안에서)."""
    if parent is not None:
        parent['suppliers'].remove(x)
    else:
        out[section].remove(x)


self_removed = 0
for c, out in results.items():
    row = companies.loc[c]
    me = {norm_name(row['stock_name']), norm_name(row['corp_name']), norm_name(row['corp_name_eng'])} - {''}
    for section, x, key, parent in list(counterparties(out)):
        if norm_name(x[key]) in me:
            remove(out, section, x, parent)
            log(c, '②', section, x[key], '', '자기 자신')
            self_removed += 1

print('자기 자신 뺌', self_removed)

자기 자신 뺌 56


## 7. ③ 지분 관계와 겹치는 기타 관계 빼기

기타 관계(`other_relations`)의 상대가 `dart_parser`의 지분 관계 표에 같은 회사로 있고, **관계 설명이 지분 관계만 말하면**(예: "신한지주의 자회사", "주요 종속회사, 지분율 100%") 뺀다. 지분 관계는 지분율·기준일이 있는 `dart_parser` 결과를 쓰는 것이 정확하다.

관계 설명에 사업 내용이 있으면 남긴다(예: 기아–동희오토 "모닝을 위탁 생산", 한독–Compass Therapeutics "임상 진행"). 고객·매입처·계약은 자회사여도 실제 거래라 처음부터 대상이 아니다.

| 지분 관계 표 | 내용 |
|---|---|
| `subsidiaries.csv` | 종속회사 |
| `associates.csv` | 관계기업·공동기업 |
| `investments.csv` | 타법인 출자 |
| `shareholders.csv` | 최대주주·특수관계인 |

In [7]:
# ── ③ 지분 관계와 겹치는 기타 관계 빼기(지분 관계는 dart_parser 표를 쓴다) ──
equity = {}
for f in ['subsidiaries', 'associates', 'investments', 'shareholders']:
    t = pd.read_csv(PARSED / f'{f}.csv', dtype=str).fillna('')
    for col in [c for c in ('name', 'alias', 'name_raw') if c in t.columns]:
        for code_, name in zip(t['stock_code'], t[col]):
            if norm_name(name):
                equity.setdefault(code_, set()).add(norm_name(name))

EQUITY_NOTE = re.compile(r'지분|자회사|종속|계열|관계기업|관계회사|공동기업|출자|주주|지배|모회사|모기업|현지법인|해외법인|투자조합|연결')
BUSINESS_NOTE = re.compile(r'공급|판매|매입|생산|개발|임상|라이선스|기술|계약|위탁|납품|공동|임대|임차|제작|시공|제공|유통|구매|수출|수입|용역|서비스')

equity_removed = 0
for c, out in results.items():
    for x in list(out['other_relations']):
        note = x['role_note']
        # 지분 표에 있는 상대이고, 설명이 지분 이야기뿐이면 뺀다(사업 내용이 있으면 남긴다)
        if (norm_name(x['counterparty']) in equity.get(c, set()) and EQUITY_NOTE.search(note)
                and not BUSINESS_NOTE.search(note)):
            out['other_relations'].remove(x)
            log(c, '③', 'other_relations', f"{x['counterparty']} ({x['role_note']})", '', '지분 관계 표에 있는 상대')
            equity_removed += 1

print('지분 관계와 겹쳐 뺌', equity_removed)

지분 관계와 겹쳐 뺌 197


## 8. ④ 국내 사업장 단위 상대 빼기

재개발·재건축 조합, 프로젝트 금융 회사(PFV), 부동산투자회사(리츠)는 공사 한 건이나 건물 하나를 위한 상대라 미국 뉴스와 이어질 일이 없다. **이름에 유형이 분명히 드러나는 것만** 뺀다. 영어 이름이 비었다는 이유만으로는 빼지 않는다(국내 상장사·협력사가 보유 종목과 이어지는 경로에 쓰일 수 있다).

대상 목록: 고객, 계약 상대, 기타 관계.

In [8]:
# ── ④ 국내 사업장 단위 상대(조합·PFV·리츠) 빼기 ──
LOCAL_RE = re.compile(r'정비사업|재개발|재건축|재정비촉진|주택조합|도시개발사업조합|피에프브이|PFV|프로젝트금융투자|위탁관리부동산투자회사')

local_removed = 0
for c, out in results.items():
    for section, x, key, parent in list(counterparties(out)):
        if section in ('customers', 'contracts', 'other_relations') and LOCAL_RE.search(x[key]):
            remove(out, section, x, parent)
            log(c, '④', section, x[key], '', '국내 사업장 단위 상대(조합·PFV·리츠)')
            local_removed += 1

print('국내 사업장 단위 상대 뺌', local_removed)
pd.DataFrame([ch for ch in changes if ch['rule'] == '④']).head(10)[['stock_name', 'section', 'before']]

국내 사업장 단위 상대 뺌 160


,stock_name,section,before
0,DL,customers,부산 촉진3구역 재개발 정비사업조합
1,DL,customers,상대원2구역 주택재개발정비사업조합
2,DL,customers,서초신동아 주택재건축 정비사업 조합
3,DL,customers,도곡개포한신 주택재건축정비사업조합
4,DL,customers,도화1구역 주택재개발정비사업조합
5,DL,customers,대전 삼성4구역 재개발정비사업조합
6,DL,customers,금송구역주택재개발정비사업조합
7,DL,customers,전도관구역주택재개발정비사업조합
8,DL,contracts,부산 촉진3구역 재개발 정비사업조합
9,DL,contracts,상대원2구역 주택재개발정비사업조합


## 9. ⑤ 이름 정리

**띄어쓰기**: 사업보고서 표는 칸을 맞추려고 글자 사이를 띄운 경우가 많다("전 장", "케 이 블"). 한 글자짜리 한글 덩어리가 **둘 이상 이어질 때만** 붙인다. "등", "및", "외"는 한 글자여도 붙이지 않는다. 예: "Naphtha (국 내)" → "Naphtha (국내)", "주석, 인동, 납 등"·"판ㆍ대 수출"은 그대로.

**중복**: 한 회사 안에서 같은 목록에 같은 항목이 두 번 있으면 처음 것만 남긴다. 같은지 보는 기준:

| 목록 | 같다고 보는 기준 |
|---|---|
| 제품, 브랜드 | 이름 |
| 원재료 | 이름과 매입처 묶음. 매입처는 원재료 안에서 이름이 같으면 하나만 |
| 고객 | 이름, 종류, 제품 |
| 경쟁사 | 이름, 제품 |
| 계약 | 상대, 종류, 내용, 금액, 기간 |
| 시장점유율, 가격 항목 | 모든 칸 |
| 기타 관계 | 상대, 관계 설명 |

In [9]:
# ── ⑤ 이름 정리: 띄어쓰기 깨짐 붙이기, 같은 항목 중복 빼기 ──
NAME_FIELDS = [('products', 'name'), ('brands', 'name'), ('materials', 'item'), ('price_items', 'item'),
               ('market_shares', 'product')]
STOP = {'등', '및', '외', '또는', '용'}


def is_single(tok):
    """한글이 한 글자뿐인 덩어리인가. 예: "전", "(국", "내)", "당(톤)". "등"·"및" 같은 말은 제외."""
    core = re.split(r'[(（]', tok.lstrip('(（'))[0]
    return (bool(re.fullmatch(r'[^\w,/·ㆍ]*[가-힣][^\w,/·ㆍ]*', core))
            and re.sub(r'[^가-힣]', '', core) not in STOP)


def fix_spacing(s):
    """한 글자 덩어리가 둘 이상 이어지면 붙이고, 바로 뒤의 한글 단어에도 붙인다."""
    toks = s.split(' ')
    out, i = [], 0
    while i < len(toks):
        j = i
        while j + 1 < len(toks) and is_single(toks[i]) and is_single(toks[j + 1]):
            j += 1
        if j == i:
            out.append(toks[i])
            i += 1
            continue
        merged = ''.join(toks[i:j + 1])
        if (j + 1 < len(toks) and re.match(r'[가-힣]', toks[j + 1]) and re.search(r'[가-힣]$', merged)
                and toks[j + 1] not in STOP):
            j += 1
            merged += toks[j]                      # "엔 진 부 품(대형)" → "엔진부품(대형)"
        out.append(merged)
        i = j + 1
    return ' '.join(out)


for t in ['전 장', '엔 진 부 품(대형+중형)', 'Naphtha (국 내)', '원 당(톤)', '주석, 인동, 납 등', '㈜풍산 판ㆍ대 수출', '병, 펫', '유제품류 수 입']:
    print(repr(t), '→', repr(fix_spacing(t)))

spacing_fixed = 0
for c, out in results.items():
    for section, key in NAME_FIELDS:
        for x in out[section]:
            new = fix_spacing(x[key])
            if new != x[key]:
                log(c, '⑤', section, x[key], new, '띄어쓰기 정리')
                x[key] = new
                spacing_fixed += 1

# 목록마다 "같은 항목"으로 볼 키
DEDUP_KEYS = {
    'products': lambda x: norm_name(x['name']),
    'brands': lambda x: norm_name(x['name']),
    'materials': lambda x: (norm_name(x['item']), tuple(sorted(norm_name(s['name']) for s in x['suppliers']))),
    'customers': lambda x: (norm_name(x['name']), x['kind'], x['product']),
    'competitors': lambda x: (norm_name(x['name']), x['product']),
    'contracts': lambda x: (norm_name(x['counterparty']), x['kind'], squash(x['subject']), squash(x['amount_text']),
                            squash(x['period_text'])),
    'market_shares': lambda x: (squash(x['product']), squash(x['market']), squash(x['share_text'])),
    'price_items': lambda x: (squash(x['item']), x['kind'], squash(x['change_text'])),
    'other_relations': lambda x: (norm_name(x['counterparty']), squash(x['role_note'])),
}

dup_removed = 0
for c, out in results.items():
    for m in out['materials']:
        seen = set()
        for s in list(m['suppliers']):
            k = norm_name(s['name'])
            if k in seen:
                m['suppliers'].remove(s)
                log(c, '⑤', 'materials.suppliers', f"{m['item']}: {s['name']}", '', '같은 원재료 안에서 중복')
                dup_removed += 1
            seen.add(k)
    for section, keyf in DEDUP_KEYS.items():
        seen = set()
        for x in list(out[section]):
            k = keyf(x)
            if k in seen:
                out[section].remove(x)
                log(c, '⑤', section, item_label(section, x), '', '같은 항목 중복')
                dup_removed += 1
            seen.add(k)

print(f'띄어쓰기 정리 {spacing_fixed}개, 중복 뺌 {dup_removed}개')

'전 장' → '전장'
'엔 진 부 품(대형+중형)' → '엔진부품(대형+중형)'
'Naphtha (국 내)' → 'Naphtha (국내)'
'원 당(톤)' → '원당(톤)'
'주석, 인동, 납 등' → '주석, 인동, 납 등'
'㈜풍산 판ㆍ대 수출' → '㈜풍산 판ㆍ대 수출'
'병, 펫' → '병, 펫'
'유제품류 수 입' → '유제품류 수입'
띄어쓰기 정리 493개, 중복 뺌 32개


## 10. 저장

- 항목을 빼서 아무 항목도 쓰지 않게 된 근거는 근거 목록에서도 지운다(기록하지 않는다).
- 최종본에는 회사마다 종목명, WI26 업종, 실행 수단, 원본 결과 파일 이름을 덧붙인다. 원본을 찾을 때 `raw_file`을 보면 된다.
- 파일 내용이 바뀌었을 때만 쓴다(바탕화면 iCloud 동기화가 사본을 만들지 않게).

In [10]:
# ── 저장: 최종본 JSON과 변경 기록 ──
def write_if_changed(path, content):
    """내용이 바뀌었을 때만 파일을 쓴다(iCloud 동기화 사본이 생기지 않게)."""
    if path.exists() and path.read_text(encoding='utf-8') == content:
        return False
    path.write_text(content, encoding='utf-8')
    return True


final = {'prompt_version': PROMPT_VERSION, 'model': MODEL, 'companies': {}}
for c, out in results.items():
    # 아무 항목도 쓰지 않게 된 근거는 지운다
    used = {x.get('evidence_id') for k in LISTS for x in out[k]}
    out['evidence'] = [e for e in out['evidence'] if e['id'] in used]
    r = ok.loc[c]
    final['companies'][c] = {'stock_name': companies.at[c, 'stock_name'], 'wi26_sector': companies.at[c, 'wi26_sector'],
                             'runner': r['runner'], 'raw_file': r['raw_file'], 'input_hash': r['input_hash'], **out}

change_df = pd.DataFrame(changes, columns=['stock_code', 'stock_name', 'rule', 'section', 'before', 'after', 'why'])
w1 = write_if_changed(FINAL / f'dart_business_{PROMPT_VERSION}.json', json.dumps(final, ensure_ascii=False, indent=1))
w2 = write_if_changed(FINAL / 'changes.csv', '﻿' + change_df.to_csv(index=False))
size = (FINAL / f'dart_business_{PROMPT_VERSION}.json').stat().st_size / 1e6
print(f'최종본 {"저장" if w1 else "바뀐 것 없음"} ({size:.1f}MB), 변경 기록 {"저장" if w2 else "바뀐 것 없음"} ({len(change_df)}줄)')

최종본 저장 (15.8MB), 변경 기록 저장 (969줄)


## 11. 결과 확인

1. 번호별 변경 건수
2. 목록별 항목 수: 원본 → 최종본
3. 최종본 점검: 모든 근거가 원문에 있는지, 모든 항목의 근거 번호가 있는지
4. 회사 하나 보기

In [11]:
# ── 확인 1~2: 규칙별 변경 건수, 목록별 항목 수(원본 → 최종본) ──
print('1) 번호별 변경 건수')
display(change_df.groupby(['rule', 'section']).size().rename('건수').reset_index())

print('2) 목록별 항목 수')
before = pd.DataFrame(raw_counts).T.sum()
after = pd.Series({k: sum(len(o[k]) for o in results.values()) for k in LISTS})
after['materials.suppliers'] = sum(len(m['suppliers']) for o in results.values() for m in o['materials'])
display(pd.DataFrame({'원본': before, '최종본': after, '뺀 것': before - after}))

1) 번호별 변경 건수


,rule,section,건수
0,①,evidence,30
1,①,price_items,1
2,②,contracts,8
3,②,customers,14
4,②,materials.suppliers,34
5,③,other_relations,197
6,④,contracts,49
7,④,customers,110
8,④,other_relations,1
9,⑤,brands,19


2) 목록별 항목 수


,원본,최종본,뺀 것
products,13743,13739,4
brands,5017,5015,2
materials,5278,5253,25
customers,3103,2979,124
competitors,1688,1688,0
market_shares,725,725,0
contracts,3584,3526,58
price_items,4565,4565,0
other_relations,1144,946,198
materials.suppliers,4107,4046,61


In [12]:
# ── 확인 3: 최종본의 근거가 모두 원문에 있고 근거 번호가 맞는지 ──
print('3) 최종본 점검')
bad_evidence = bad_id = 0
for c, out in results.items():
    src = squash(texts[c])
    ids = {e['id'] for e in out['evidence']}
    bad_evidence += sum(squash(e['text']) not in src for e in out['evidence'])
    bad_id += sum(x.get('evidence_id') not in ids for k in LISTS for x in out[k])
print(f'원문에 없는 근거 {bad_evidence}개, 없는 근거 번호를 쓴 항목 {bad_id}개')
assert bad_evidence == 0 and bad_id == 0

3) 최종본 점검


원문에 없는 근거 0개, 없는 근거 번호를 쓴 항목 0개


In [13]:
# ── 확인 4: 회사 하나의 변경 기록 ──
print('4) 회사 하나 보기: 변경 기록')
SHOW = '103140'   # 풍산
display(change_df[change_df['stock_code'] == SHOW])
show = final['companies'][SHOW]
print(show['stock_name'], {k: len(show[k]) for k in LISTS})

4) 회사 하나 보기: 변경 기록


,stock_code,stock_name,rule,section,before,after,why
19,103140,풍산,①,evidence,"| 방산부문 | 제품 | 정밀부품 | 국가방위산업 | "" | 12,610 | 13.2% | 13,521 | 33.6% ...","| 방산부문 | 제품 | 정밀부품 | 국가방위산업 | "" | 12,610 | 13.2% | 13,521 | 33.6% ...",원문과 다른 근거를 원문으로 바꿈(일치율 0.98)
20,103140,풍산,①,evidence,"| 신동부문 | 제품 등 | 스크랩 등 | 산업용소재 | - | 436,470 | 5.8% | 331,594 | 4.0...","| 신동부문 | 제품 등 | 스크랩 등 | 산업용소재 | - | 436,470 | 5.8% | 331,594 | 4.0...",원문과 다른 근거를 원문으로 바꿈(일치율 0.98)


풍산 {'products': 19, 'brands': 7, 'materials': 17, 'customers': 0, 'competitors': 0, 'market_shares': 0, 'contracts': 0, 'price_items': 30, 'other_relations': 0}
